# 03. 심화: Factual Priming, Hallucination, Test-Time Selection

목표: 관련 사실이 정답 회상을 돕는 효과와, 중간 사실 환각이 최종 답을 망치는 효과를 작은 데이터로 구현합니다.

실행 방법: 모든 셀을 순서대로 실행합니다. 외부 패키지는 필요하지 않습니다.

## 1. 질문과 reasoning trace 샘플

각 trace에는 중간 사실 목록, 사실 검증 결과, 최종 답이 들어 있습니다. 실제 논문은 검색 가능한 verifier로 대규모 trace를 감사했습니다.

In [ ]:
samples = [
    {
        "question": "Who was the 10th King of Nepal?",
        "answer": "Birendra Bir Bikram Shah Dev",
        "trace_id": "t1",
        "facts": [
            ("Prithvi Narayan Shah unified Nepal.", True),
            ("Mahendra was a predecessor of Birendra.", True),
        ],
        "final": "Birendra Bir Bikram Shah Dev",
    },
    {
        "question": "Who was the 10th King of Nepal?",
        "answer": "Birendra Bir Bikram Shah Dev",
        "trace_id": "t2",
        "facts": [("Jitari Malla was the 10th Shah king.", False)],
        "final": "Jitari Malla",
    },
    {
        "question": "What year was Mary Engle Pennington inducted into the National Inventors Hall of Fame?",
        "answer": "2002",
        "trace_id": "t3",
        "facts": [
            ("Mary Engle Pennington worked on food refrigeration.", True),
            ("She is associated with public health and food preservation.", True),
        ],
        "final": "2002",
    },
    {
        "question": "What year was Mary Engle Pennington inducted into the National Inventors Hall of Fame?",
        "answer": "2002",
        "trace_id": "t4",
        "facts": [("She won a Nobel Prize in 1937.", False)],
        "final": "1937",
    },
    {
        "question": "What year was Mary Engle Pennington inducted into the National Inventors Hall of Fame?",
        "answer": "2002",
        "trace_id": "t5",
        "facts": [],
        "final": "2001",
    },
]


def normalize(text):
    return " ".join(text.lower().split())


def final_is_correct(sample):
    return normalize(sample["final"]) == normalize(sample["answer"])


def has_any_fact(sample):
    return len(sample["facts"]) > 0


def has_hallucination(sample):
    return any(not is_true for _fact, is_true in sample["facts"])


for sample in samples:
    print(sample["trace_id"], "facts=", len(sample["facts"]), "hallucinated=", has_hallucination(sample), "correct=", final_is_correct(sample))

## 2. Clean trace와 hallucinated trace 비교

논문의 큰 메시지는 중간 사실이 깨끗한 trace가 최종 답도 더 잘 맞힌다는 것입니다.

In [ ]:
def accuracy(rows):
    if not rows:
        return 0.0
    return sum(final_is_correct(row) for row in rows) / len(rows)


with_facts = [sample for sample in samples if has_any_fact(sample)]
clean = [sample for sample in with_facts if not has_hallucination(sample)]
hallucinated = [sample for sample in with_facts if has_hallucination(sample)]

print("all traces accuracy:        ", round(accuracy(samples), 3))
print("with facts accuracy:        ", round(accuracy(with_facts), 3))
print("clean fact traces accuracy: ", round(accuracy(clean), 3))
print("hallucinated traces acc:    ", round(accuracy(hallucinated), 3))

## 3. Test-time selection

여러 trace가 있을 때, 사실을 포함하고 환각이 없는 trace를 먼저 선택하는 전략을 구현합니다.

In [ ]:
def select_trace_for_question(rows, criterion):
    candidates = list(rows)
    if criterion == "regular":
        # 무필터 baseline입니다. 여기서는 임의의 마지막 샘플을 선택한다고 둡니다.
        return candidates[-1]
    if criterion == "only_facts":
        fact_rows = [row for row in candidates if has_any_fact(row)]
        return fact_rows[0] if fact_rows else candidates[0]
    if criterion == "only_correct_facts":
        clean_rows = [row for row in candidates if has_any_fact(row) and not has_hallucination(row)]
        return clean_rows[0] if clean_rows else candidates[0]
    raise ValueError(criterion)


questions = sorted(set(sample["question"] for sample in samples))
for criterion in ["regular", "only_facts", "only_correct_facts"]:
    selected = []
    for question in questions:
        rows = [sample for sample in samples if sample["question"] == question]
        selected.append(select_trace_for_question(rows, criterion))
    print(criterion, "selected=", [row["trace_id"] for row in selected], "accuracy=", round(accuracy(selected), 3))

## 4. Process reward 설계 스케치

최종 답만 보상하면 모델이 우연히 맞힌 trace와 사실적으로 좋은 trace를 구분하기 어렵습니다. 아래는 중간 사실의 존재와 정확성에 점수를 주는 간단한 process reward입니다.

In [ ]:
def process_reward(sample):
    reward = 0.0
    if final_is_correct(sample):
        reward += 1.0
    if has_any_fact(sample):
        reward += 0.2
    if has_hallucination(sample):
        reward -= 0.8
    else:
        reward += 0.3
    return reward


ranked = sorted(samples, key=process_reward, reverse=True)
for sample in ranked:
    print(
        sample["trace_id"],
        "reward=", round(process_reward(sample), 2),
        "correct=", final_is_correct(sample),
        "hallucinated=", has_hallucination(sample),
    )